In [38]:
import os
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

# Helper function to display images in the notebook without color issues (OpenCV uses BGR, Matplotlib uses RGB)
def display_image(img_array, size=(10, 10)):
    img_rgb = cv2.cvtColor(img_array, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=size)
    plt.imshow(img_rgb)
    plt.axis('off')
    plt.show()

print("✅ Setup complete. Ready for inference.")

✅ Setup complete. Ready for inference.


In [39]:
MODEL_PATH = "../models/best_yolo_v2.pt"

# Load the model
model = YOLO(MODEL_PATH)

# Let's define your class names so the output makes sense to us
CLASS_NAMES = {
    0: "bin_caged",
    1: "bin_elevated",
    2: "bin_ground",
    3: "trap_object"
}

print(f"✅ Model loaded successfully from {MODEL_PATH}")

✅ Model loaded successfully from ../models/best_yolo_v2.pt


In [40]:
# Point this to an image in your test set
TEST_IMAGE_PATH = "../test_data/images/img_001.png"

# Run inference! (conf=0.5 means we only want predictions it is at least 50% sure about)
results = model.predict(source=TEST_IMAGE_PATH, conf=0.5)

# Extract the first result (since we only passed one image)
result = results[0]

# YOLO has a built-in .plot() function that draws the boxes for you
annotated_img = result.plot()

print("🔍 YOLO Raw Inference Output:")
display_image(annotated_img)

RuntimeError: CUDA error: unspecified launch failure
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [33]:
import numpy as np

# YOLO stores the original image array right in the result object (in BGR format)
original_img = result.orig_img 
img_h, img_w = original_img.shape[:2]

# Store our processed, padded crops here to pass to the ViT in Stage 2
vit_inputs = []

print("✂️ Processing YOLO detections and extracting 50% Halo crops...\n")

for box in result.boxes:
    # 1. Extract raw data
    class_id = int(box.cls[0].item())
    confidence = float(box.conf[0].item())
    class_name = CLASS_NAMES[class_id]
    
    # 2. THE SINK LOGIC: Ignore trap objects immediately
    if class_name == "trap_object":
        print(f"🛡️ Ignored a {class_name} (Conf: {confidence:.2f})")
        continue
        
    # 3. Get original bounding box coordinates
    x1, y1, x2, y2 = map(float, box.xyxy[0].tolist())
    bbox_w = x2 - x1
    bbox_h = y2 - y1
    
    # 4. 50% Halo Math (Add 25% padding on all 4 sides)
    pad_x = bbox_w * 1.5
    pad_y = bbox_h * 1.5
    
    # Calculate new bounds and clamp to image edges to prevent crashing
    crop_x1 = int(max(0, x1 - pad_x))
    crop_y1 = int(max(0, y1 - pad_y))
    crop_x2 = int(min(img_w, x2 + pad_x))
    crop_y2 = int(min(img_h, y2 + pad_y))
    
    # 5. Extract the padded image slice
    crop_img = original_img[crop_y1:crop_y2, crop_x1:crop_x2]
    
    if crop_img.size > 0:
        vit_inputs.append({
            "type": class_name,
            "confidence": confidence,
            "crop": crop_img
        })
        print(f"🎯 Extracted Target: {class_name} | Padded Size: {crop_img.shape[1]}x{crop_img.shape[0]}")
    else:
        print(f"⚠️ Warning: Invalid crop calculated for {class_name}")

# 6. Visualize the extracted ViT inputs side-by-side
print("\n" + "="*50)
if vit_inputs:
    # Dynamically scale the plot based on how many bins were found
    fig, axes = plt.subplots(1, len(vit_inputs), figsize=(5 * len(vit_inputs), 5))
    
    # Handle the case where there's only one crop (matplotlib won't return an array of axes)
    if len(vit_inputs) == 1:
        axes = [axes]
        
    for ax, item in zip(axes, vit_inputs):
        # Convert OpenCV's BGR format to Matplotlib's RGB for accurate colors
        img_rgb = cv2.cvtColor(item["crop"], cv2.COLOR_BGR2RGB)
        ax.imshow(img_rgb)
        ax.set_title(f"{item['type']}\nYOLO Conf: {item['confidence']:.2f}", fontsize=12, fontweight='bold')
        ax.axis('off')
        
    plt.tight_layout()
    plt.show()
else:
    print("No valid target bins found in this image. (Only background or traps).")

✂️ Processing YOLO detections and extracting 50% Halo crops...


No valid target bins found in this image. (Only background or traps).


In [34]:
import torch
from torchvision import transforms
from transformers import SwinForImageClassification
from PIL import Image

# 1. Setup & Load Model (Forcing CPU to comply with competition rules)
device = torch.device("cpu")
print(f"💻 Processing on: {device}\n")

# NOTE: Update this path to point to your newly trained best.pth file
VIT_WEIGHTS_PATH = "../models/best_vit.pth"

print("🧠 Loading Fine-Tuned Stage 2 Swin Transformer...")

# Initialize the architecture exactly as we did in training
vit_model = SwinForImageClassification.from_pretrained(
    "microsoft/swin-base-patch4-window7-224",
    num_labels=2,
    id2label={0: "action_required", 1: "no_action"},
    label2id={"action_required": 0, "no_action": 1},
    ignore_mismatched_sizes=True
)

# Load your custom trained weights safely onto the CPU
vit_model.load_state_dict(torch.load(VIT_WEIGHTS_PATH, map_location=device))
vit_model.to(device)
vit_model.eval() # CRITICAL: Set to evaluation mode to disable dropout layers!

# 2. Define the exact same transforms used during validation
vit_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 3. Stage 2: The Contextual Observer (Process Crops)
print("\n🔍 Analyzing Crops for Spills/Overflows...")

# We will store the binary DMC decisions for each individual bin here
bin_decisions = []

if not vit_inputs:
    print("⚠️ No valid inputs passed from Stage 1.")
else:
    for i, item in enumerate(vit_inputs):
        # The ViT was trained on PIL images, but OpenCV loads them as NumPy BGR arrays.
        # We must convert BGR -> RGB -> PIL Image to match the training data perfectly.
        img_rgb = cv2.cvtColor(item["crop"], cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(img_rgb)
        
        # Apply transforms and add the batch dimension [1, Channels, Height, Width]
        input_tensor = vit_transforms(pil_img).unsqueeze(0).to(device)
        
        # Forward pass without calculating gradients (saves memory & speeds up inference)
        with torch.no_grad():
            outputs = vit_model(input_tensor).logits
            predicted_class_id = torch.argmax(outputs, dim=1).item()
            
        # Map the ViT output ID to the string label
        predicted_label = vit_model.config.id2label[predicted_class_id]
        
        # Translate the label to the final DMC integer logic
        if predicted_label == "action_required":
            dmc_score = 1
            icon = "🚨"
        else:
            dmc_score = 0
            icon = "✅"
            
        bin_decisions.append(dmc_score)
        print(f"Crop {i+1} [{item['type']}]: {predicted_label.upper()} {icon} --> Intimate DMC: {dmc_score}")

# 4. Stage 3: The Deterministic Aggregation Head
print("\n" + "="*50)
print("⚙️ Stage 3: Final Decision Aggregation")
print("="*50)

final_decision = 0

if len(bin_decisions) == 0:
    print("Reason: No authorized bins were found in the frame. (DMC is not concerned).")
    final_decision = 0
else:
    # The OR Logic: If ANY bin requires action, the whole image requires action.
    if any(decision == 1 for decision in bin_decisions):
        print("Reason: At least one authorized bin in the frame has an active spill or overflow.")
        final_decision = 1
    else:
        print("Reason: All detected bins are clean and below the overflow threshold.")
        final_decision = 0

print(f"\n🏆 FINAL OUTPUT SCORE: {final_decision}")

💻 Processing on: cpu

🧠 Loading Fine-Tuned Stage 2 Swin Transformer...


You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.
Loading weights: 100%|██████████| 449/449 [00:00<00:00, 14484.48it/s]
SwinForImageClassification LOAD REPORT from: microsoft/swin-base-patch4-window7-224
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([2])            
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 1024]) vs model:torch.Size([2, 1024])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.
/tmp/ipykernel_25753/572624024.py:25: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickl


🔍 Analyzing Crops for Spills/Overflows...
⚠️ No valid inputs passed from Stage 1.

⚙️ Stage 3: Final Decision Aggregation
Reason: No authorized bins were found in the frame. (DMC is not concerned).

🏆 FINAL OUTPUT SCORE: 0
